<a href="https://colab.research.google.com/github/mbaker21231/MicroII-Sandbox/blob/main/Hopenhayn2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code to simulate Hopenhayn model

A first requirement is a utility function to make a continuous distribution into a grid. Here it is:

In [2]:
#Packages

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

I think it is a good idea to define all the parameters we are using in one place, and basically keep track of them on the basis of whether they are globals or locals, and experimental values.

In [3]:
N     = 100
MU    = -.25
RHO   = .85
SIGMA = .40
K_E   = 2

# Global parameters

phi_c = 1
beta  = .05
alpha = .5

# model parameters

tolerance = 1e-10

In [4]:
def tauchen(N, mu, rho, sigma, n_std=4):
    z = np.linspace(mu - n_std * sigma / np.sqrt(1 - rho**2),
                     mu + n_std * sigma / np.sqrt(1 - rho**2), N)
    step = (z[1] - z[0])
    P = np.zeros((N, N))

    for j in range(N):
        for k in range(N):
            if k == 0:
                P[j, k] = norm.cdf((z[k] - rho * z[j] + step / 2) / sigma)
            elif k == N-1:
                P[j, k] = 1 - norm.cdf((z[k] - rho * z[j] - step / 2) / sigma)
            else:
                P[j, k] = (norm.cdf((z[k] - rho * z[j] + step / 2) / sigma) -
                           norm.cdf((z[k] - rho * z[j] - step / 2) / sigma))

    return z, P


Note that we can also use this distribution to recover a cumulative unconditional distribution, which is useful for initial productivity draws:

In [5]:
def init_dist(N, mu, rho, sigma, n_std=4):

    tauch = tauchen(N, mu, rho, sigma, n_std)
    probs = np.sum(tauch[1], axis=0)/np.sum(tauch[1])
    vals = tauch[0]

    return vals, probs

## Aspects of the model

Each firm has a flow profit function of the form:
$$
\pi(z) = \tilde z n^\alpha - Wn
$$

where $\tilde z$ is the (exponentiated) skill level $z$, $\tilde z=e^z$. Flow profits are acheived by choosing $n$, labor, to maximize th above:

$$
 \alpha \tilde z n^{\alpha -1}-W \quad \rightarrow\quad n^*(z,w) = \left(\frac{\alpha \tilde z}{W}\right)^\frac{1}{1-\alpha}
$$

Here is a function that returns, for a given wage and skill level, profits and labor demand:

In [6]:
def prof_lab(z, W):

  z_tilde = np.exp(z)
  n_sta   = ( alpha * z_tilde / W)**(1/(1-alpha))
  profs   = z_tilde*n_sta**alpha - W*alpha

  return profs, n_sta

## Present value of a firm

The following bit of code essentially iterates the value function, taking into account that the firm's valuation changes as a result of possible changes in $z$, the skill level of the firm.

In [7]:
def val_fun(z, p, W, max_iter=3000, tol=1e-10, noisy=False):

  v = np.zeros((len(z), 1))

  for i in range(max_iter):

    profs = prof_lab(z, W)[0]
    profs = np.reshape(profs, (len(z), 1))
    vnew = np.maximum( 0, profs - phi_c + (1-beta)* p @ v)
    if np.max(abs(vnew-v)<tol):
      break

  if noisy:
    print("Iterations: ", i)
    print("Maximum value: ", np.max(vnew))
    print("Average value: ", np.mean(vnew))
    print("Minimum value: ", np.min(vnew))

  return vnew

## Computing an equilibrium wage

Let's first take a stab at computing an equilibrium wage. Intuitively, we want the wage to be such that the supply of labor is equal to the total demand for labor. Where firms that do not produce exit the market.

In [8]:
Z, P = tauchen(N, MU, RHO, SIGMA)       # Skills and firms
W = .74                                   # Initial wage guess
M = 2                                   # Mass of firms
V = val_fun(Z, P, W, noisy=True)        # Value functions at skills
beven = np.argmax(V > tolerance)        # Index of first nonzero value
pi, n = prof_lab(Z[beven], W)           # Profits, labor of active firms

np.sum(n), beven


Iterations:  0
Maximum value:  176.76988011827711
Average value:  14.763083538379727
Minimum value:  0.0


(1.004479713642167, 60)

So, we are beginning to get the idea - we can increase the wage and get a resulting number of active firms. Now, to get dynamics, we need to do get some initial distribution of firms. We have:

So, if I am understanding things correctly, we have to determine the measure of entrants. As Edmond has it, we have a law of motion for entrants as:

$$
\mu_{t+1}([0,z']) = \int F(z'|z)\mathbf{1}[z\geq z_t^*]\mu_t(dz)+m_{t+1}G(z')
$$

That looks difficult, but Edmond argues that this can be discretized to a grid with a certain number of elements, and we have:

$$
\mathbf{\mu_{t+1}} = \Psi_t\mathbf{\mu_t} + m_{t+1}\mathbf{g}
$$

If I'm reading everything correctly, the matrix $\Psi_t$ is basically the non-zero elements of the transition matrix. $m$ is a scalar, according to Edmonds. We note that what we want is the steady-state distribution of firms:

$$
\mathbf{\mu} = \Psi \mathbf{\mu} + m \mathbf{g}
$$

Let's try to solve this by iteration.

In [9]:
Z_init, P_init = init_dist(N, MU, RHO, SIGMA)


And really, the key thing is that the only thing that really matters for firms - the only way they interact  with one another - is through the wage. So, my guess is that we need to do the following loop:
1. Given an equilibrium wage, we compute which firms are active using profits and values.
2. Given a mass of entrants, and the fraction of active firms, we compute the long run distribution of firms.
3. Given the long run distribution of firms, we compute the equilibrium wage.

Repeat until convergence. With that in mind, we can start off a distribution of skill, so we have:

In [10]:
Z, P = tauchen(N, MU, RHO, SIGMA)       # Skills and firms
W = .74                                 # Initial wage guess
M = 2                                   # Mass of firms

In [11]:
#### Iteration one.
V = val_fun(Z, P, W, noisy=True)        # Value functions at skills
beven = np.argmax(V > tolerance)        # Index of first nonzero value

Iterations:  0
Maximum value:  176.76988011827711
Average value:  14.763083538379727
Minimum value:  0.0


In [12]:
# Iteration one continued... active firms
beven = np.argmax(V > tolerance)        # Index of first nonzero value
pi, n = prof_lab(Z[beven], W)           # Profits, labor of active firms
print("Cutoff:", beven)

Cutoff: 60


So, the 60th position is the first place where firms are active. It follows that all firms with z below z[60] are inactive, so we can say that these firms exit. What is the best way to do this? From a formal perspective, it seems like the best thing to do would be to replace the upper part of the matrix with an identity matrix and see what happens...

In [1]:
def replace_upper_with_identity(A, K):

    N = A.shape[0]
    A[:K, :K] = np.eye(K)  # Replace top-left KxK block with an identity matrix
    return A

In [18]:
Psi  = replace_upper_with_identity(P, beven)
mask = (V > 0).astype(int)

In [25]:
G = M * P_init * mask

In [28]:
def iterate_until_convergence(P, G, max_iters=1000, tol=1e-9):

    M = np.zeros(P.shape[0])  # Initialize M with M0
    for t in range(max_iters):
        M_next = P @ M + G  # Matrix-vector multiplication step
        if np.allclose(M_next, M, atol=tol):  # Check for convergence
            return M_next, t+1
        M = M_next  # Update M

    return M, max_iters  # Return M after max iterations


In [29]:
Mss = iterate_until_convergence(Psi, M * P_init * mask)

In [30]:
Mss

(array([[1.39191685e+49, 4.75714056e+48, 5.96481170e+48, ...,
         7.67076885e+48, 6.25667191e+48, 1.98917928e+49],
        [3.96020840e+49, 1.35347654e+49, 1.69707676e+49, ...,
         2.18244669e+49, 1.78011529e+49, 5.65950794e+49],
        [1.10218460e+50, 3.76692550e+49, 4.72321576e+49, ...,
         6.07407210e+49, 4.95432427e+49, 1.57512481e+50],
        ...,
        [6.38250236e+62, 2.18134157e+62, 2.73510769e+62, ...,
         3.51735812e+62, 2.86893741e+62, 9.12119242e+62],
        [6.27575417e+62, 2.14485835e+62, 2.68936265e+62, ...,
         3.45852984e+62, 2.82095406e+62, 8.96863928e+62],
        [6.17517339e+62, 2.11048295e+62, 2.64626055e+62, ...,
         3.40310039e+62, 2.77574296e+62, 8.82489995e+62]]),
 1000)

In [31]:
Psi

array([[1.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [1.20865776e-43, 8.82854662e-43, 7.13874301e-42, ...,
        3.71470139e-02, 3.15054760e-02, 1.09476165e-01],
       [1.96289095e-44, 1.46654113e-43, 1.20976092e-42, ...,
        4.19480437e-02, 3.62947199e-02, 1.35899444e-01],
       [3.13431250e-45, 2.39512055e-44, 2.01560535e-43, ...,
        4.65725867e-02, 4.11085059e-02, 1.66387386e-01]])